# 🛰️ Project Chakravyuh — Satellite Tank Inventory CV Pipeline
### Floating-Roof Tank Detection & Barrel Fill-Volume Estimation (Proof-of-Concept)

> **⚠️ Reality Ledger (Tier C — Modeled):** This is a **reproducible proof-of-concept pipeline**, not a trained production model. It demonstrates the full flow — fetch a Planet Labs scene → detect floating-roof tanks → estimate fill % from shadow geometry → barrels. When live imagery or an API key is unavailable it runs on a **synthetic scene**, so the pipeline is fully reproducible offline. The interpretable **Hough-circle + shadow-geometry** baseline below is the fallback; a production build fine-tunes the imported **YOLOv8-Segmentation** model on labeled tank masks.

Target sites: **Jamnagar & Vadinar** crude-storage complexes (3 m Planet Labs / Sentinel-2 imagery).

In [ ]:
# Step 1: Install Required Dependencies
!pip install -q ultralytics opencv-python-headless matplotlib requests numpy pandas planet


In [ ]:
# Step 2: Configure API Key & Coordinates
import os
import requests
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Read the key from the environment — never hard-code secrets in a committed notebook.
PLANET_API_KEY = os.environ.get("PLANET_API_KEY", "")
JAMNAGAR_COORDS = {"lat": 22.4707, "lng": 70.0577}
VADINAR_COORDS = {"lat": 22.4411, "lng": 69.6738}

print("✅ Planet API key loaded from environment." if PLANET_API_KEY
      else "ℹ️ No PLANET_API_KEY set — running on the synthetic-scene fallback.")

In [ ]:
# Step 3: Fetch Satellite Scene Tile or Generate High-Res Synthetic Scene
def fetch_planet_scene(lat, lng, zoom=17):
    url = f"https://tiles.planet.com/basemaps/v1/planet-tiles/global_monthly_2024_01_mosaic/gmap/17/100000/100000.png?api_key={PLANET_API_KEY}"
    res = requests.get(url)
    if res.status_code == 200:
        nparr = np.frombuffer(res.content, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        canvas = np.full((640, 640, 3), 35, dtype=np.uint8)
        tanks = [(180, 180, 70), (450, 180, 70), (180, 450, 70), (450, 450, 70)]
        for cx, cy, r in tanks:
            cv2.circle(canvas, (cx, cy), r, (120, 130, 140), -1)
            cv2.circle(canvas, (cx + 15, cy + 15), r - 12, (50, 55, 60), -1)
            cv2.circle(canvas, (cx, cy), r, (200, 200, 200), 2)
        return canvas

scene_img = fetch_planet_scene(JAMNAGAR_COORDS["lat"], JAMNAGAR_COORDS["lng"])
plt.figure(figsize=(8, 8))
plt.imshow(scene_img)
plt.title("Satellite Imagery — Jamnagar Crude Storage Complex")
plt.axis("off")
plt.show()


In [ ]:
# Step 4: Segmentation & Tank Fill Volume Calculation
def estimate_tank_capacity(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=100,
        param1=50, param2=30, minRadius=40, maxRadius=100
    )
    results = []
    output_img = img.copy()
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        for idx, (x, y, r) in enumerate(circles, 1):
            mask = np.zeros_like(gray)
            cv2.circle(mask, (x, y), r, 255, -1)
            tank_pixels = np.sum(mask > 0)
            shadow_mask = (gray < 70) & (mask > 0)
            shadow_pixels = np.sum(shadow_mask)
            fill_pct = max(10.0, min(100.0, 100.0 * (1.0 - (shadow_pixels / max(1, tank_pixels)))))
            max_cap_bbls = 500000
            current_bbls = max_cap_bbls * (fill_pct / 100.0)
            results.append({"tank_id": f"TANK_{idx:02d}", "fill_pct": round(fill_pct, 1), "barrels": int(current_bbls)})
            cv2.circle(output_img, (x, y), r, (0, 255, 128), 3)
            cv2.putText(output_img, f"{fill_pct:.0f}%", (x - 20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return output_img, results

annotated_img, tank_inventory = estimate_tank_capacity(scene_img)
plt.figure(figsize=(10, 10))
plt.imshow(annotated_img)
plt.title("Chakravyuh Computer Vision Output — Detected Tanks & Fill Levels")
plt.axis("off")
plt.show()
print("📊 DEDUCED TANK INVENTORY RESULTS:")
for t in tank_inventory:
    print(f"  • {t['tank_id']}: {t['fill_pct']}% Full ({t['barrels']:,} Barrels)")